# DarCloud QuranChain — Deploy Runner
Bismillah. Executes Wrangler deployment commands in sequence: migration check, migration apply, worker deploy, and endpoint tests.

## 1. Check D1 Migration Status

In [1]:
import subprocess

print("=" * 60)
print("STEP 1: Checking D1 migration status...")
print("=" * 60)

try:
    result = subprocess.run(
        "npx wrangler d1 migrations list DB --remote",
        cwd="/workspaces/quranchain1",
        capture_output=True,
        text=True,
        timeout=120,
        shell=True,
    )
    print("STDOUT:")
    print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    print(f"Return code: {result.returncode}")
except subprocess.TimeoutExpired:
    print("ERROR: Command timed out after 120 seconds")
except Exception as e:
    print(f"ERROR: {e}")

STEP 1: Checking D1 migration status...
STDOUT:

 ⛅️ wrangler 4.73.0
───────────────────
Resource location: remote 

✅ No migrations to apply!

Return code: 0


## 2. Apply Pending D1 Migrations

In [2]:
print("=" * 60)
print("STEP 2: Applying pending D1 migrations...")
print("=" * 60)

try:
    result = subprocess.run(
        "npx wrangler d1 migrations apply DB --remote",
        cwd="/workspaces/quranchain1",
        capture_output=True,
        text=True,
        timeout=120,
        shell=True,
        input="y\n",
    )
    print("STDOUT:")
    print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    print(f"Return code: {result.returncode}")
except subprocess.TimeoutExpired:
    print("ERROR: Command timed out after 120 seconds — continuing to next step")
except Exception as e:
    print(f"ERROR: {e} — continuing to next step")

STEP 2: Applying pending D1 migrations...
STDOUT:

 ⛅️ wrangler 4.73.0
───────────────────
Resource location: remote 

✅ No migrations to apply!

Return code: 0


## 3. Deploy Cloudflare Worker

In [3]:
print("=" * 60)
print("STEP 3: Deploying Cloudflare Worker...")
print("=" * 60)

try:
    result = subprocess.run(
        "npx wrangler deploy",
        cwd="/workspaces/quranchain1",
        capture_output=True,
        text=True,
        timeout=120,
        shell=True,
    )
    print("STDOUT:")
    print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    print(f"Return code: {result.returncode}")
except subprocess.TimeoutExpired:
    print("ERROR: Command timed out after 120 seconds — continuing to endpoint tests")
except Exception as e:
    print(f"ERROR: {e} — continuing to endpoint tests")

STEP 3: Deploying Cloudflare Worker...
STDOUT:

 ⛅️ wrangler 4.73.0
───────────────────
Total Upload: 1088.35 KiB / gzip: 199.06 KiB
Worker Startup Time: 12 ms
Your Worker has access to the following bindings:
Binding                           Resource         
env.DB (openapi-template-db)      D1 Database      

Uploaded quranchain1 (20.03 sec)
Deployed quranchain1 triggers (10.71 sec)
  darcloud.host/* (zone name: darcloud.host)
  *.darcloud.host/* (zone name: darcloud.host)
  darcloud.net/* (zone name: darcloud.net)
  *.darcloud.net/* (zone name: darcloud.net)
Current Version ID: e409714a-a68a-417a-b94e-6160eedcd319

Return code: 0


## 4. Test Fixed Endpoints

In [4]:
print("=" * 60)
print("STEP 4: Testing fixed endpoints...")
print("=" * 60)

# Test /telecom/towers
print("\n--- GET https://darcloud.host/telecom/towers ---")
try:
    result = subprocess.run(
        "curl -s https://darcloud.host/telecom/towers | head -c 500",
        cwd="/workspaces/quranchain1",
        capture_output=True,
        text=True,
        timeout=120,
        shell=True,
    )
    print("STDOUT:")
    print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    print(f"Return code: {result.returncode}")
except subprocess.TimeoutExpired:
    print("ERROR: Command timed out after 120 seconds")
except Exception as e:
    print(f"ERROR: {e}")

# Test /telecom/signal-map
print("\n--- GET https://darcloud.host/telecom/signal-map ---")
try:
    result = subprocess.run(
        "curl -s https://darcloud.host/telecom/signal-map | head -c 500",
        cwd="/workspaces/quranchain1",
        capture_output=True,
        text=True,
        timeout=120,
        shell=True,
    )
    print("STDOUT:")
    print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    print(f"Return code: {result.returncode}")
except subprocess.TimeoutExpired:
    print("ERROR: Command timed out after 120 seconds")
except Exception as e:
    print(f"ERROR: {e}")

print("\n" + "=" * 60)
print("Alhamdulillah — deployment sequence complete.")
print("=" * 60)

STEP 4: Testing fixed endpoints...

--- GET https://darcloud.host/telecom/towers ---
STDOUT:
404 Not Found
Return code: 0

--- GET https://darcloud.host/telecom/signal-map ---
STDOUT:
404 Not Found
Return code: 0

Alhamdulillah — deployment sequence complete.


In [8]:
import subprocess

endpoints = [
    "https://darcloud.host/isp/towers",
    "https://darcloud.host/isp/signal-map",
    "https://darcloud.host/isp/dashboard",
    "https://darcloud.host/isp/plans",
    "https://darcloud.host/isp/coverage",
    "https://darcloud.host/isp/wifi-directory",
    "https://darcloud.host/telecom/plans",
    "https://darcloud.host/telecom/dashboard",
    "https://darcloud.host/api/companies",
    "https://darcloud.host/mesh/nodes",
]

lines = ["=== DarCloud Endpoint Test ==="]
for url in endpoints:
    result = subprocess.run(
        ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", url],
        capture_output=True, text=True, timeout=15
    )
    code = result.stdout.strip()
    path = url.replace("https://darcloud.host", "")
    lines.append(f"{path} → {code}")

with open("/workspaces/quranchain1/endpoint-test-results.txt", "w") as f:
    f.write("\n".join(lines) + "\n")
print("Results written to endpoint-test-results.txt")

Results written to endpoint-test-results.txt


In [ ]:
import subprocess

cwd = "/workspaces/quranchain1"

commands = [
    ["git", "add", "-A"],
    ["git", "status", "--short"],
    ["git", "commit", "-m", "fix: remove hardcoded creds from PowerShell scripts, fix open proxy vuln, fix ISP endpoint SQL bugs, fix migration 0009 idempotency"],
    ["git", "push", "origin", "codespace-humble-goggles-x5vgxvjj679qcv79w"],
]

for cmd in commands:
    print(f"\n{'='*60}")
    print(f"$ {' '.join(cmd)}")
    print('='*60)
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"Exit code: {result.returncode}")
    if result.returncode != 0 and cmd[1] != "status":
        print("Command failed, stopping.")
        break